# Notebook 1 — ChEMBL Data Fetch
**Dataset Creation, Curation, and Analysis of GPCR Receptor Targets**

This notebook fetches raw Ki binding activity for the configured target set from
ChEMBL, applies query-time filters, and emits a clean CSV for Notebook 2.

### Design principles
- This notebook makes **no descriptor decisions** — it only fetches and filters.
- Every filter logs a before/after count, so the curation cascade
  (*"50 configured → N retained"*) is **generated by code**, not asserted by hand.
- **Phase I–IV only.** Preclinical / null `max_phase` compounds are dropped at
  fetch time, per advisor instruction.
- Dedup (max pKi per compound–target) and salt/peptide removal happen in
  **Notebook 2**, not here. Row counts printed here are therefore *pre-dedup*.

### Requirements
```
pip install chembl_webresource_client pandas tqdm
```
Needs network access to `www.ebi.ac.uk`. Will not run in a sandbox.

### Outputs (in `data/`)
| file | contents |
|------|----------|
| `raw_activities_unfiltered.csv` | everything pulled (audit trail) |
| `raw_filtered.csv` | input to Notebook 2 |
| `fetch_report.csv` | per-target drop table |


## 0. Configuration
Single source of truth for filter parameters and the target set.

In [1]:
import os
import time
from collections import OrderedDict

import pandas as pd

OUTDIR = "data"
os.makedirs(OUTDIR, exist_ok=True)

# --- Filter parameters: change here, re-run, done ---
MW_MAX          = 700.0
ALLOWED_PHASES  = {1, 2, 3, 4}      # preclinical / null EXCLUDED (advisor)
STANDARD_TYPE   = "Ki"
TARGET_ORGANISM = "Homo sapiens"
RETRY_LIMIT     = 3
RETRY_SLEEP     = 5                  # seconds between retries

In [2]:
TARGET_UNIPROT_MAP = OrderedDict([
    ('Adenosine A1','P30542'),('Adenosine A2A','P29274'),
    ('Adrenergic alpha1D','P25100'),('Adrenergic alpha2A','P08913'),
    ('Adrenergic alpha2B','P18089'),('Adrenergic alpha2C','P18825'),
    ('Adrenergic beta1','P08588'),('Adrenergic beta2','P07550'),
    ('Adrenergic beta3','P13945'),('Angiotensin AT2','P50052'),
    ('Bradykinin B2','P30411'),('Calcitonin','P30988'),
    ('Cannabinoid CB1','P21554'),('Chemokine CCR2B','P41597'),
    ('Chemokine CCR4','P51679'),('Chemokine CCR5','P51681'),
    ('Cholecystokinin CCKA','P32238'),('Dopamine D1','P21728'),
    ('Dopamine D2L','P14416'),('Dopamine D3','P35462'),
    ('Dopamine D4.2','P21917'),('Endothelin ETA','P25101'),
    ('Estrogen ERalpha','P03372'),('Estrogen ERbeta','Q92731'),
    ('Leukotriene B4','Q9NPC1'),('Melanocortin MC3','P32245'),
    ('Melanocortin MC4','P32246'),('Melanocortin MC5','P33032'),
    ('Muscarinic M1','P11229'),('Muscarinic M2','P08172'),
    ('Muscarinic M3','P20309'),('Muscarinic M4','P08173'),
    ('Muscarinic M5','P08912'),('Neuropeptide Y1','P25929'),
    ('Neuropeptide Y2','P49146'),('Opiate delta','P41143'),
    ('Opiate kappa','P41145'),('Opiate mu','P35372'),
    ('Serotonin 5-HT1A','P08908'),('Serotonin 5-HT1B','P28222'),
    ('Serotonin 5-HT2A','P28223'),('Serotonin 5-HT2B','P41595'),
    ('Serotonin 5-HT2C','P28335'),('Serotonin 5-HT3','P46098'),
    ('Serotonin 5-HT4','Q13639'),('Serotonin 5-HT6','P50406'),
    ('Sigma1','Q99720'),('Tachykinin NK1','P25103'),
    ('Tachykinin NK2','P21452'),('Vasopressin V1A','P37288'),
])

RECEPTOR_FAMILIES = {
    'Serotonin':['Serotonin 5-HT1A','Serotonin 5-HT1B','Serotonin 5-HT2A','Serotonin 5-HT2B',
                 'Serotonin 5-HT2C','Serotonin 5-HT3','Serotonin 5-HT4','Serotonin 5-HT6'],
    'Adrenergic':['Adrenergic alpha1D','Adrenergic alpha2A','Adrenergic alpha2B',
                  'Adrenergic alpha2C','Adrenergic beta1','Adrenergic beta2','Adrenergic beta3'],
    'Dopamine':['Dopamine D1','Dopamine D2L','Dopamine D3','Dopamine D4.2'],
    'Muscarinic':['Muscarinic M1','Muscarinic M2','Muscarinic M3','Muscarinic M4','Muscarinic M5'],
    'Opiate':['Opiate delta','Opiate kappa','Opiate mu'],
    'Adenosine':['Adenosine A1','Adenosine A2A'],
    'Cannabinoid':['Cannabinoid CB1'],
    'Chemokine':['Chemokine CCR2B','Chemokine CCR4','Chemokine CCR5'],
    'Melanocortin':['Melanocortin MC3','Melanocortin MC4','Melanocortin MC5'],
    'Tachykinin':['Tachykinin NK1','Tachykinin NK2'],
    'Estrogen':['Estrogen ERalpha','Estrogen ERbeta'],
    'Neuropeptide Y':['Neuropeptide Y1','Neuropeptide Y2'],
    'Sigma':['Sigma1'],
    'Angiotensin':['Angiotensin AT2'],
    'Bradykinin':['Bradykinin B2'],
    'Calcitonin':['Calcitonin'],
    'Cholecystokinin':['Cholecystokinin CCKA'],
    'Endothelin':['Endothelin ETA'],
    'Leukotriene':['Leukotriene B4'],
    'Vasopressin':['Vasopressin V1A'],
}

Consistency checks — every target belongs to exactly one family.

In [3]:
_fam_targets = [t for ts in RECEPTOR_FAMILIES.values() for t in ts]
assert set(_fam_targets) == set(TARGET_UNIPROT_MAP), (
    "Mismatch between TARGET_UNIPROT_MAP and RECEPTOR_FAMILIES:\n"
    f"  in map not in families: {set(TARGET_UNIPROT_MAP) - set(_fam_targets)}\n"
    f"  in families not in map: {set(_fam_targets) - set(TARGET_UNIPROT_MAP)}"
)
assert len(_fam_targets) == len(set(_fam_targets)), "A target appears in >1 family"

TARGET_TO_FAMILY = {t: f for f, ts in RECEPTOR_FAMILIES.items() for t in ts}
N_CONFIGURED = len(TARGET_UNIPROT_MAP)
print(f"Configured targets: {N_CONFIGURED}")
print(f"Families: {len(RECEPTOR_FAMILIES)}")

Configured targets: 50
Families: 20


## 1. ChEMBL client + helper functions
`resolve_target` maps each UniProt accession to a single-protein human ChEMBL
target ID (logging failures). `fetch_activities` pulls Ki binding rows with
retries on transient API errors.

In [4]:
from chembl_webresource_client.new_client import new_client

target_client   = new_client.target
activity_client = new_client.activity
molecule_client = new_client.molecule


def resolve_target(uniprot_id):
    """UniProt accession -> ChEMBL target_chembl_id (single human protein).
    Returns (chembl_id, pref_name) or (None, reason)."""
    try:
        hits = list(target_client.filter(
            target_components__accession=uniprot_id,
            target_type="SINGLE PROTEIN",
        ).only("target_chembl_id", "pref_name", "organism"))
    except Exception as e:
        return None, f"query_error: {e}"
    if not hits:
        return None, "no_chembl_target"
    human = [h for h in hits if h.get("organism") == TARGET_ORGANISM]
    chosen = human[0] if human else hits[0]
    return chosen["target_chembl_id"], chosen.get("pref_name", "")


def fetch_activities(chembl_id):
    """Pull Ki binding activities for one target, with retries."""
    fields = [
        "molecule_chembl_id", "canonical_smiles", "standard_type",
        "standard_relation", "standard_value", "standard_units",
        "pchembl_value", "assay_type", "target_chembl_id",
    ]
    for attempt in range(1, RETRY_LIMIT + 1):
        try:
            qs = activity_client.filter(
                target_chembl_id=chembl_id,
                standard_type=STANDARD_TYPE,
                assay_type="B",
                target_organism=TARGET_ORGANISM,
            ).only(fields)
            return pd.DataFrame(list(qs))
        except Exception as e:
            if attempt == RETRY_LIMIT:
                print(f"    FAILED after {RETRY_LIMIT} attempts: {e}")
                return pd.DataFrame()
            print(f"    attempt {attempt} failed ({e}); retry in {RETRY_SLEEP}s")
            time.sleep(RETRY_SLEEP)
    return pd.DataFrame()

/home/dagma/miniconda3/envs/allchem/lib/python3.10/site-packages/chembl_webresource_client/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


## 2. Fetch loop
Iterates all configured targets. Per target it records raw count, valid-pChEMBL
count, and with-SMILES count into `report_rows` for the final drop table.

In [5]:
report_rows = []
all_frames  = []

for i, (name, uniprot) in enumerate(TARGET_UNIPROT_MAP.items(), 1):
    print(f"[{i:>2}/{N_CONFIGURED}] {name}  ({uniprot})")
    chembl_id, info = resolve_target(uniprot)

    rec = {
        "target": name, "uniprot": uniprot, "family": TARGET_TO_FAMILY[name],
        "chembl_id": chembl_id, "resolve_note": info if chembl_id is None else "ok",
        "n_raw": 0, "n_has_pchembl": 0, "n_has_smiles": 0,
        "n_mw_pass": None, "n_phase_pass": None, "retained_pre_dedup": 0,
    }

    if chembl_id is None:
        print(f"    UNRESOLVED: {info}")
        report_rows.append(rec)
        continue

    df = fetch_activities(chembl_id)
    rec["n_raw"] = len(df)
    if df.empty:
        print("    0 activities")
        report_rows.append(rec)
        continue

    df["target"]  = name
    df["uniprot"] = uniprot
    df["family"]  = TARGET_TO_FAMILY[name]

    df["pchembl_value"]  = pd.to_numeric(df["pchembl_value"], errors="coerce")
    df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")

    df = df[df["pchembl_value"].notna()]
    rec["n_has_pchembl"] = len(df)
    df = df[df["canonical_smiles"].notna() & (df["canonical_smiles"] != "")]
    rec["n_has_smiles"] = len(df)
    rec["retained_pre_dedup"] = len(df)

    print(f"    raw={rec['n_raw']}  valid_pchembl={rec['n_has_pchembl']}  "
          f"with_smiles={rec['n_has_smiles']}")
    all_frames.append(df)
    report_rows.append(rec)
    time.sleep(0.3)   # be polite to the API

if not all_frames:
    raise SystemExit("No data fetched — check network / client install.")

raw = pd.concat(all_frames, ignore_index=True)
raw.to_csv(os.path.join(OUTDIR, "raw_activities_unfiltered.csv"), index=False)
print(f"\nUnfiltered pull saved: {len(raw)} rows across "
      f"{raw['target'].nunique()} targets")

[ 1/50] Adenosine A1  (P30542)
    raw=6839  valid_pchembl=4985  with_smiles=4985
[ 2/50] Adenosine A2A  (P29274)
    raw=8508  valid_pchembl=6071  with_smiles=6071
[ 3/50] Adrenergic alpha1D  (P25100)
    raw=2318  valid_pchembl=1386  with_smiles=1386
[ 4/50] Adrenergic alpha2A  (P08913)
    raw=1971  valid_pchembl=1019  with_smiles=1012
[ 5/50] Adrenergic alpha2B  (P18089)
    raw=1611  valid_pchembl=712  with_smiles=709
[ 6/50] Adrenergic alpha2C  (P18825)
    raw=1784  valid_pchembl=865  with_smiles=862
[ 7/50] Adrenergic beta1  (P08588)
    raw=1357  valid_pchembl=386  with_smiles=386
[ 8/50] Adrenergic beta2  (P07550)
    raw=1859  valid_pchembl=796  with_smiles=796
[ 9/50] Adrenergic beta3  (P13945)
    raw=1255  valid_pchembl=357  with_smiles=357
[10/50] Angiotensin AT2  (P50052)
    raw=1164  valid_pchembl=246  with_smiles=246
[11/50] Bradykinin B2  (P30411)
    raw=1235  valid_pchembl=316  with_smiles=316
[12/50] Calcitonin  (P30988)
    raw=886  valid_pchembl=13  with_smiles

## 3. Molecule metadata + MW / phase filters
`max_phase` and `full_mwt` live on the molecule record, so we batch-fetch
molecule metadata for the unique compounds, merge it in, then filter.

> **Note:** `max_phase` is assigned at the *molecule* level in ChEMBL, not
> per-target. A compound approved for any indication carries phase 4 even
> against a target where it was only a research tool. This matches the prior
> draft's behaviour and is worth one sentence in Methods.

In [6]:
unique_ids = sorted(raw["molecule_chembl_id"].dropna().unique().tolist())
print(f"Fetching molecule metadata for {len(unique_ids)} unique compounds...")

mol_meta = {}
CHUNK = 50
for start in range(0, len(unique_ids), CHUNK):
    chunk = unique_ids[start:start + CHUNK]
    for attempt in range(1, RETRY_LIMIT + 1):
        try:
            mols = molecule_client.filter(
                molecule_chembl_id__in=chunk
            ).only(["molecule_chembl_id", "max_phase",
                    "molecule_properties", "molecule_type"])
            for m in mols:
                props = m.get("molecule_properties") or {}
                mw = props.get("full_mwt")
                mol_meta[m["molecule_chembl_id"]] = {
                    "max_phase": m.get("max_phase"),
                    "mw": pd.to_numeric(mw, errors="coerce") if mw is not None else None,
                    "molecule_type": m.get("molecule_type"),
                }
            break
        except Exception as e:
            if attempt == RETRY_LIMIT:
                print(f"    metadata chunk {start} failed: {e}")
            else:
                time.sleep(RETRY_SLEEP)
    if (start // CHUNK) % 10 == 0:
        print(f"    {min(start + CHUNK, len(unique_ids))}/{len(unique_ids)}")

meta_df = pd.DataFrame.from_dict(mol_meta, orient="index")
meta_df.index.name = "molecule_chembl_id"
meta_df = meta_df.reset_index()
raw = raw.merge(meta_df, on="molecule_chembl_id", how="left")
raw["max_phase"] = pd.to_numeric(raw["max_phase"], errors="coerce")

Fetching molecule metadata for 48900 unique compounds...
    50/48900
    550/48900
    1050/48900
    1550/48900
    2050/48900
    2550/48900
    3050/48900
    3550/48900
    4050/48900
    4550/48900
    5050/48900
    5550/48900
    6050/48900
    6550/48900
    7050/48900
    7550/48900
    8050/48900
    8550/48900
    9050/48900
    9550/48900
    10050/48900
    10550/48900
    11050/48900
    11550/48900
    12050/48900
    12550/48900
    13050/48900
    13550/48900
    14050/48900
    14550/48900
    15050/48900
    15550/48900
    16050/48900
    16550/48900
    17050/48900
    17550/48900
    18050/48900
    18550/48900
    19050/48900
    19550/48900
    20050/48900
    20550/48900
    21050/48900
    21550/48900
    22050/48900
    22550/48900
    23050/48900
    23550/48900
    24050/48900
    24550/48900
    25050/48900
    25550/48900
    26050/48900
    26550/48900
    27050/48900
    27550/48900
    28050/48900
    28550/48900
    29050/48900
    29550/48900
    30

In [7]:
mw_pass    = raw[raw["mw"] < MW_MAX]
phase_pass = mw_pass[mw_pass["max_phase"].isin(ALLOWED_PHASES)]

mw_counts    = mw_pass.groupby("target")["molecule_chembl_id"].count().to_dict()
phase_counts = phase_pass.groupby("target")["molecule_chembl_id"].count().to_dict()
for rec in report_rows:
    rec["n_mw_pass"]    = mw_counts.get(rec["target"], 0)
    rec["n_phase_pass"] = phase_counts.get(rec["target"], 0)

print(f"After MW < {MW_MAX}:     {len(mw_pass)} rows")
print(f"After Phase I-IV only:  {len(phase_pass)} rows")

filtered = phase_pass.copy()
filtered.to_csv(os.path.join(OUTDIR, "raw_filtered.csv"), index=False)

After MW < 700.0:     98858 rows
After Phase I-IV only:  6533 rows


## 4. Curation summary — *50 configured → N retained*
Generated entirely from the logged counts. `fetch_report.csv` is the audit
trail to show your advisor / examiner exactly why each dropped target dropped.

In [8]:
report = pd.DataFrame(report_rows)
report["retained_final"] = report["target"].map(
    filtered.groupby("target")["molecule_chembl_id"].count()
).fillna(0).astype(int)
report["dropped_target"] = report["retained_final"] == 0
report.to_csv(os.path.join(OUTDIR, "fetch_report.csv"), index=False)

n_retained = (report["retained_final"] > 0).sum()
dropped = report[report["retained_final"] == 0][["target", "resolve_note", "n_raw"]]

print("=" * 60)
print("CURATION SUMMARY")
print("=" * 60)
print(f"Targets configured: {N_CONFIGURED}")
print(f"Targets retained:   {n_retained}")
print(f"Targets dropped:    {N_CONFIGURED - n_retained}")
if len(dropped):
    print("\nDropped targets (and why):")
    for _, r in dropped.iterrows():
        why = r["resolve_note"] if r["resolve_note"] != "ok" else \
              ("no rows passed filters" if r["n_raw"] else "0 activities")
        print(f"  - {r['target']:<22} {why}")
print(f"\nRows (compound-target pairs) after filters: {len(filtered)}")
print(f"Unique compounds: {filtered['molecule_chembl_id'].nunique()}")
print("\nNOTE: dedup + salt/peptide removal happen in Notebook 2.")
print("Counts above are PRE-dedup.")
print("=" * 60)

CURATION SUMMARY
Targets configured: 50
Targets retained:   47
Targets dropped:    3

Dropped targets (and why):
  - Calcitonin             no rows passed filters
  - Leukotriene B4         no rows passed filters
  - Neuropeptide Y1        no rows passed filters

Rows (compound-target pairs) after filters: 6533
Unique compounds: 722

NOTE: dedup + salt/peptide removal happen in Notebook 2.
Counts above are PRE-dedup.


In [9]:
# Quick look at the per-target report
report[["target", "family", "n_raw", "n_has_pchembl",
        "n_mw_pass", "n_phase_pass", "retained_final"]]

,target,family,n_raw,n_has_pchembl,n_mw_pass,n_phase_pass,retained_final
0,Adenosine A1,Adenosine,6839,4985,4902,134,134
1,Adenosine A2A,Adenosine,8508,6071,6012,170,170
2,Adrenergic alpha1D,Adrenergic,2318,1386,1381,156,156
3,Adrenergic alpha2A,Adrenergic,1971,1019,997,198,198
4,Adrenergic alpha2B,Adrenergic,1611,712,694,167,167
5,Adrenergic alpha2C,Adrenergic,1784,865,855,165,165
6,Adrenergic beta1,Adrenergic,1357,386,386,74,74
7,Adrenergic beta2,Adrenergic,1859,796,735,121,121
8,Adrenergic beta3,Adrenergic,1255,357,353,32,32
9,Angiotensin AT2,Angiotensin,1164,246,181,4,4
